In [1]:
# from scripts.csv_processing import normalize_propforms
import pandas as pd
import re

## Rejoigner les fichiers pertinents pour sauvegarder les instances standards pour le projet

In [6]:
PATH_TO_DATA = '../data/external/ls-fr-V3.1'

FILES_REQUIRED = {
    'nodes': '01-lsnodes.csv',
    'entries': '02-lsentries.csv',
    'ex-rel': '18-lsex-rel.csv',
    'examples': '17-lsex.csv',
    'propforms': '11-lspropform-rel.csv',
    'pos': '06-lsgramcharac-rel.csv',
    'lfs': '15-lslf-rel.csv'
    } # les fichiers pertinents pour établir un tableau des FL exemples
def get_csv(filename, path=PATH_TO_DATA, separator='\t'):
    return pd.read_csv(f'{path}/{filename}', sep=separator)

### Combiner fichier des nodes et des entries

In [ ]:
nodes = get_csv(FILES_REQUIRED['nodes'])
nodes = nodes.rename(columns={'id': 'node_id', 'entry':'entry_id'})
print(nodes.shape)
nodes.head()

(29783, 8)


,node_id,entry_id,lexnum,status,%,update_date,update_time,lexname
0,ls:fr:node:26162,ls:fr:entry:26164,I.1,2,100,2021-03-13,09:45:08,<span class='namingform'>à</span><span class='...
1,ls:fr:node:26163,ls:fr:entry:26165,1,2,100,2017-03-07,08:28:24,<span class='namingform'>à propos</span><span ...
2,ls:fr:node:26164,ls:fr:entry:26166,II.2,2,100,2017-12-03,21:05:52,<span class='namingform'>abandonner</span><spa...
3,ls:fr:node:26166,ls:fr:entry:26168,I,2,100,2017-03-25,20:39:46,<span class='namingform'>abîme</span><span cla...
4,ls:fr:node:26167,ls:fr:entry:26169,I,2,100,2014-05-15,10:56:19,<span class='namingform'>abîmer</span><span cl...


In [8]:
entries = get_csv(FILES_REQUIRED['entries'])
# traiter les formes des lexies
entries['entry_name'] = entries.apply(lambda x: (str(x['addtoname']) + x['name']) if pd.notna(x['addtoname']) else x['name'], axis=1) # combiner les lexies avec les 'addtoname' puis les mettre dans 'entry_name'
entries = entries.rename(columns={'id': 'entry_id'})
print(entries.shape)
entries.head()

(18892, 8)


,entry_id,addtoname,name,subscript,superscript,status,%,entry_name
0,ls:fr:entry:26164,NaN,à,NaN,NaN,1,100,à
1,ls:fr:entry:26165,NaN,à propos,NaN,NaN,2,100,à propos
2,ls:fr:entry:26166,NaN,abandonner,NaN,NaN,2,100,abandonner
3,ls:fr:entry:26167,NaN,abeille,NaN,NaN,1,100,abeille
4,ls:fr:entry:26168,NaN,abîme,NaN,NaN,2,100,abîme


In [9]:
# Convert entry_id to match type between nodes and entries
nodes_entry = nodes.merge(entries[['entry_name', 'entry_id']], 
						 left_on='entry_id', 
						 right_on='entry_id', 
						 how='left')
print(nodes_entry.shape)
nodes_entry.head()

(29783, 9)


,node_id,entry_id,lexnum,status,%,update_date,update_time,lexname,entry_name
0,ls:fr:node:26162,ls:fr:entry:26164,I.1,2,100,2021-03-13,09:45:08,<span class='namingform'>à</span><span class='...,à
1,ls:fr:node:26163,ls:fr:entry:26165,1,2,100,2017-03-07,08:28:24,<span class='namingform'>à propos</span><span ...,à propos
2,ls:fr:node:26164,ls:fr:entry:26166,II.2,2,100,2017-12-03,21:05:52,<span class='namingform'>abandonner</span><spa...,abandonner
3,ls:fr:node:26166,ls:fr:entry:26168,I,2,100,2017-03-25,20:39:46,<span class='namingform'>abîme</span><span cla...,abîme
4,ls:fr:node:26167,ls:fr:entry:26169,I,2,100,2014-05-15,10:56:19,<span class='namingform'>abîmer</span><span cl...,abîmer


In [10]:
lfs = pd.read_csv('../data/interim/lfs_names_modified.csv')
lfs_names = lfs.set_index('lf_id')['lf_name'].to_dict()

In [11]:
def normalize_propforms(propform):
    """
    Fix some common bugs in propositional forms.
    Propositional forms starting with a "!" need to be validated.
    These "!" are removed and the propositional form is considered valid.
    """
    propform = re.sub(r"'", r'’', propform)
    propform = re.sub(r' +,', r',', propform)
    propform = re.sub(r'(\w)~', r'\1 ~', propform)
    propform = re.sub(r'~(\w)', r'~ \1', propform)
    propform = re.sub(r'\s\s+', r' ', propform)
    propform = re.sub(r'^! ?', r'', propform)
    return propform.strip()

propforms = get_csv(FILES_REQUIRED['propforms'])
propforms.propform = propforms.propform.apply(normalize_propforms)
propforms.set_index('node', inplace=True)
print(propforms.shape)
propforms.head()

(23399, 4)


,propform,tildevalue,%,actantslist
node,,,,
ls:fr:node:26162,[$1] ~ $2,NaN,100,"($1=X,$2=Y)"
ls:fr:node:26163,[$1] ~ de $2,NaN,100,"($1=X,$2=Y)"
ls:fr:node:26166,~,NaN,100,()
ls:fr:node:26167,$1 ~ $2,abîme,100,"($1=X,$2=Y)"
ls:fr:node:26168,$1 ~ $2 à $3,abonne,100,"($1=X,$2=Y,$3=Z)"


In [12]:
propforms[propforms.propform.str.contains('$')]

,propform,tildevalue,%,actantslist
node,,,,
ls:fr:node:26162,[$1] ~ $2,NaN,100,"($1=X,$2=Y)"
ls:fr:node:26163,[$1] ~ de $2,NaN,100,"($1=X,$2=Y)"
ls:fr:node:26166,~,NaN,100,()
ls:fr:node:26167,$1 ~ $2,abîme,100,"($1=X,$2=Y)"
ls:fr:node:26168,$1 ~ $2 à $3,abonne,100,"($1=X,$2=Y,$3=Z)"
...,...,...,...,...
ls:fr:node:56948,$1 ~,NaN,100,($1=X)
ls:fr:node:56949,$1 ~,NaN,100,($1=X)
ls:fr:node:56950,~ utilisée par $1 pour $2,NaN,100,"($1=X,$2=Y)"


In [13]:
def replace_actants(prop_f:str, actant_l:str):
    actants = actant_l.strip("()")
    mapping = dict(re.findall(r'(\$\d+)=(\w+)', actants))
    act_nums = [act_num for act_num in re.findall(r'\$\d+', prop_f)]
    for act_num in act_nums:
        if act_num in mapping:
            prop_f = prop_f.replace(act_num, mapping[act_num])
    
    return prop_f

def antantialize(df: pd.DataFrame):
    df = df.copy()
    df['actant_form'] = df.apply(lambda x: replace_actants(x['propform'], x['actantslist']), axis=1)
    df['actant_form'] = df.apply(lambda x: x['actant_form'].replace('~', str(x['tildevalue'])) if pd.notna(x['tildevalue']) else x['actant_form'], axis=1)
    return df
    
propforms = antantialize(propforms)

In [14]:
propforms.head()

,propform,tildevalue,%,actantslist,actant_form
node,,,,,
ls:fr:node:26162,[$1] ~ $2,NaN,100,"($1=X,$2=Y)",[X] ~ Y
ls:fr:node:26163,[$1] ~ de $2,NaN,100,"($1=X,$2=Y)",[X] ~ de Y
ls:fr:node:26166,~,NaN,100,(),~
ls:fr:node:26167,$1 ~ $2,abîme,100,"($1=X,$2=Y)",X abîme Y
ls:fr:node:26168,$1 ~ $2 à $3,abonne,100,"($1=X,$2=Y,$3=Z)",X abonne Y à Z


In [15]:
propforms.shape

(23399, 5)

## POS


In [21]:
preposition_id = ["ls:fr:gc:121", "ls:fr:gc:64", "ls:fr:gc:214"]
pos = get_csv(FILES_REQUIRED['pos'])
nodes_entry['POS'] = nodes_entry['node_id'].map(pos.set_index('node')['POS'])
nodes_entry[nodes_entry['POS'].isin(preposition_id)]

,node_id,entry_id,lexnum,status,%,update_date,update_time,lexname,entry_name,POS
0,ls:fr:node:26162,ls:fr:entry:26164,I.1,2,100,2021-03-13,09:45:08,<span class='namingform'>à</span><span class='...,à,ls:fr:gc:64
172,ls:fr:node:26341,ls:fr:entry:26343,I.1,2,100,2017-12-07,06:54:15,<span class='namingform'>après</span><span cla...,après,ls:fr:gc:64
564,ls:fr:node:26745,ls:fr:entry:26747,NaN,2,100,2014-07-15,10:49:26,<span class='namingform'>chez</span>,chez,ls:fr:gc:64
717,ls:fr:node:26899,ls:fr:entry:26901,II,2,100,2020-06-23,15:41:25,<span class='namingform'>contre</span><span cl...,contre,ls:fr:gc:64
811,ls:fr:node:26993,ls:fr:entry:26995,I.1,2,100,2017-06-11,14:00:01,<span class='namingform'>dans</span><span clas...,dans,ls:fr:gc:64
...,...,...,...,...,...,...,...,...,...,...
27344,ls:fr:node:54440,ls:fr:entry:44123,NaN,3,100,2016-12-05,08:57:47,<span class='namingform'>grâce</span><span cla...,grâce,ls:fr:gc:64
27934,ls:fr:node:55057,ls:fr:entry:27282,II.b,3,100,2020-08-13,15:31:24,<span class='namingform'>en</span><span class=...,en,ls:fr:gc:64
28179,ls:fr:node:55309,ls:fr:entry:29481,II.2,3,100,2018-01-22,16:52:37,<span class='namingform'>sous</span><span clas...,sous,ls:fr:gc:64
28192,ls:fr:node:55322,ls:fr:entry:27080,II,3,100,2018-01-18,09:44:50,<span class='namingform'>derrière</span><span ...,derrière,ls:fr:gc:64


## exemples et nodes

In [12]:
exemple_rel = get_csv(FILES_REQUIRED['ex-rel'])
# un node peut avoir plusieurs exemples
print(exemple_rel.shape)
exemple_rel.head()

(51056, 5)


,node,example,occurrence,position,%
0,ls:fr:node:26162,ls:fr:ex:4328,"66,67;",0,100
1,ls:fr:node:26162,ls:fr:ex:337,"161,162;",1,100
2,ls:fr:node:26162,ls:fr:ex:331,"37,38;",2,100
3,ls:fr:node:26162,ls:fr:ex:3964,"26,27;",3,100
4,ls:fr:node:26162,ls:fr:ex:3348,"53,54;",4,100


In [13]:
def get_keyword(df: pd.DataFrame, ex_col:str = "content", kw_seg_col: str = "occurrence"):
    """
    Get the keyword from the example.
    The keyword coresponds to the 
    """
    df

In [14]:
exemples = get_csv(FILES_REQUIRED['examples'])
print(exemples.shape)
exemples.head()

(32026, 8)


,id,source,status,content,title,authors,location,date
0,ls:fr:ex:13,ls:fr:exsrc:1,1,<html><body><p>Nous voulions kidnapper le géra...,Voyage au bout de la révolution : de Pékin à S...,"Brière-Blanchet, Claire",p. 116,2009
1,ls:fr:ex:14,ls:fr:exsrc:1,1,<html><body><p>Son patron lui a dit qu’il étai...,Journal 1977-1990,"Lagarce, Jean-Luc",p. 325,2007
2,ls:fr:ex:15,ls:fr:exsrc:1,1,<html><body><p>Elle a dit qu’elle t’avait cher...,Les Russkoffs,"Cavanna, François",p. 373,1979
3,ls:fr:ex:16,ls:fr:exsrc:2,1,<html><body><p>Les ravisseurs ont profité d’un...,NaN,NaN,http://www.yozone.fr/spip.php?article4460,02/2008
4,ls:fr:ex:17,ls:fr:exsrc:2,1,<html><body><p>Pour s’assurer que Jake lui rem...,NaN,NaN,http://tetelle43.blogs.allocine.fr/?blog=tetel...,02/2008


In [15]:
exemples.value_counts("status")

status
1    29985
0     2041
Name: count, dtype: int64

In [16]:
exemples=exemples.rename(columns={'id': 'ex_id'})
exemples.head()

,ex_id,source,status,content,title,authors,location,date
0,ls:fr:ex:13,ls:fr:exsrc:1,1,<html><body><p>Nous voulions kidnapper le géra...,Voyage au bout de la révolution : de Pékin à S...,"Brière-Blanchet, Claire",p. 116,2009
1,ls:fr:ex:14,ls:fr:exsrc:1,1,<html><body><p>Son patron lui a dit qu’il étai...,Journal 1977-1990,"Lagarce, Jean-Luc",p. 325,2007
2,ls:fr:ex:15,ls:fr:exsrc:1,1,<html><body><p>Elle a dit qu’elle t’avait cher...,Les Russkoffs,"Cavanna, François",p. 373,1979
3,ls:fr:ex:16,ls:fr:exsrc:2,1,<html><body><p>Les ravisseurs ont profité d’un...,NaN,NaN,http://www.yozone.fr/spip.php?article4460,02/2008
4,ls:fr:ex:17,ls:fr:exsrc:2,1,<html><body><p>Pour s’assurer que Jake lui rem...,NaN,NaN,http://tetelle43.blogs.allocine.fr/?blog=tetel...,02/2008


In [17]:
ex_content = exemples.set_index('ex_id')['content'].to_dict()
print(len(ex_content))
ex_content

32026


{'ls:fr:ex:13': '<html><body><p>Nous voulions kidnapper le gérant et le juger. Nous avions entamé une campagne de propagande contre «\xa0le gérant fasciste de Seyssinet\xa0».</p></body></html>',
 'ls:fr:ex:14': '<html><body><p>Son patron lui a dit qu’il était le seul de ses collaborateurs que les «\xa0chiites\xa0» ne garderaient pas plus de trois jours s’ils venaient à le kidnapper et qu’ils verseraient même une rançon pour le rendre.</p></body></html>',
 'ls:fr:ex:15': '<html><body><p>Elle a dit qu’elle t’avait cherché longtemps, que des troufions l’avaient kidnappée mais qu’elle s’était échappée, et alors elle est retournée là où vous étiez, et puis elle a parcouru le pays dans tous les sens en demandant après toi, et finalement, voilà, elle essayait de retarder son rapatriement le plus possible dans l’espoir que tu finirais par arriver...</p></body></html>',
 'ls:fr:ex:16': '<html><body><p>Les ravisseurs ont profité d’une absence de vigilance de la mère pour kidnapper la petite Aman

In [18]:
exemple_rel['content'] = exemple_rel['example'].map(ex_content)
# exemple_rel = exemple_rel.merge(nodes_entry[['node_id', 'entry_name']], 
#                   left_on='node', 
#                   right_on='node_id', 
#                   how='left', 
#                   suffixes=('', '_entry'))
# exemple_rel = exemple_rel[["node", "example", "entry_name", "content", "occurrence", "position"]]
print(exemple_rel.shape)

# We only keep the first example for each node, witch is the most relevant one
exemple_rel = exemple_rel[exemple_rel['position'] == 0].copy()
print(exemple_rel.shape)
exemple_rel.head()

(51056, 6)
(27811, 6)


,node,example,occurrence,position,%,content
0,ls:fr:node:26162,ls:fr:ex:4328,"66,67;",0,100,<html><body><p>C’est comme de changer d’avion ...
9,ls:fr:node:26163,ls:fr:ex:4628,"62,70;",0,100,"<html><body><p>Le docteur Ricci, que maman m’a..."
11,ls:fr:node:26164,ls:fr:ex:1323,"48,57;",0,100,<html><body><p>Pourtant un jour elle partit et...
12,ls:fr:node:26166,ls:fr:ex:10465,"203,208;",0,100,<html><body><p>De nombreux Constantinois allai...
13,ls:fr:node:26167,ls:fr:ex:11675,"150,155;",0,100,<html><body><p>« On remplit au fur et à mesure...


In [19]:
def highlight_by_positions(sentence: str, positions_str: str, wrapper=("【", "】")):
    if pd.isna(positions_str) or not positions_str.strip():
        return sentence

    # 处理位置字符串
    positions = []
    for pos in positions_str.strip(";").split(";"):
        if "," in pos:
            try:
                start, end = map(int, pos.split(","))
                positions.append((start - 1, end - 1))  # 包含终止字符
            except:
                continue

    # 从后往前插入，防止坐标错乱
    positions.sort(reverse=True, key=lambda x: x[0])

    for start, end in positions:
        if 0 <= start < end <= len(sentence):
            sentence = sentence[:start] + wrapper[0] + sentence[start:end] + wrapper[1] + sentence[end:]
    return sentence

In [20]:
exemple_rel['ex_highlighted'] = exemple_rel.apply(
    lambda row: highlight_by_positions(row['content'], row['occurrence']),
    axis=1
)
bop = "<html><body><p>"
eop = "</p></body></html>"
exemple_rel['ex_highlighted'] = exemple_rel['ex_highlighted'].str.replace(bop, '').str.replace(eop, '')
exemple_rel_dict = exemple_rel.set_index('node')["ex_highlighted"].to_dict()
exemple_rel.head().to_csv('../data/interim/exemples_highlighted.csv', index=False)

In [21]:
lfs_rel = get_csv(FILES_REQUIRED['lfs'])
print(lfs_rel.columns)
print(lfs_rel.shape)
lfs_instances = lfs_rel[['lf', 'source', 'target','form','merged']].copy()
lfs_instances.head()

Index(['source', 'lf', 'target', 'form', 'separator', 'merged',
       'syntacticframe', 'constraint', 'position'],
      dtype='object')
(65489, 9)


,lf,source,target,form,merged
0,ls:fr:lf:3,ls:fr:node:26162,ls:fr:node:43890,NaN,0
1,ls:fr:lf:3,ls:fr:node:26162,ls:fr:node:41336,NaN,0
2,ls:fr:lf:3,ls:fr:node:26162,ls:fr:node:29809,NaN,0
3,ls:fr:lf:3,ls:fr:node:26163,ls:fr:node:32392,NaN,0
4,ls:fr:lf:3,ls:fr:node:26163,ls:fr:node:32518,NaN,0


In [22]:
# combine
lfs_instances['lf_name'] = lfs_instances['lf'].map(lfs_names)
lfs_instances = lfs_instances[['lf', 'lf_name', 'source', 'target', 'form', 'merged']]
lfs_instances.head()

,lf,lf_name,source,target,form,merged
0,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:43890,NaN,0
1,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:41336,NaN,0
2,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:29809,NaN,0
3,ls:fr:lf:3,Syn_∩,ls:fr:node:26163,ls:fr:node:32392,NaN,0
4,ls:fr:lf:3,Syn_∩,ls:fr:node:26163,ls:fr:node:32518,NaN,0


In [23]:
# if source == target
lfs_instances[lfs_instances['source'] == lfs_instances['target']]

,lf,lf_name,source,target,form,merged
883,ls:fr:lf:845,ClausCaus_1Pred,ls:fr:node:26370,ls:fr:node:26370,« ~ ! »,0
2396,ls:fr:lf:40,S_2,ls:fr:node:26668,ls:fr:node:26668,~,0
3463,ls:fr:lf:385,Result_2,ls:fr:node:26854,ls:fr:node:26854,NaN,1
4623,ls:fr:lf:40,S_2,ls:fr:node:27109,ls:fr:node:27109,~,0
4769,ls:fr:lf:68,S_med,ls:fr:node:27142,ls:fr:node:27142,~,0
...,...,...,...,...,...,...
64227,ls:fr:lf:40,S_2,ls:fr:node:56430,ls:fr:node:56430,~,0
64252,ls:fr:lf:40,S_2,ls:fr:node:56440,ls:fr:node:56440,~,0
64550,ls:fr:lf:40,S_2,ls:fr:node:56523,ls:fr:node:56523,~,0
64990,ls:fr:lf:31,S_1,ls:fr:node:56703,ls:fr:node:56703,~,0


In [26]:
lfs_instances.shape

(65489, 7)

In [27]:
# 添加每个function的keywords和values
node_id_to_entry_name = nodes_entry.set_index('node_id')['entry_name'].to_dict()
lfs_instances['keywords'] = lfs_instances['source'].map(node_id_to_entry_name)
lfs_instances['values'] = lfs_instances['target'].map(node_id_to_entry_name)
lfs_instances["kw_context"] = lfs_instances["source"].map(exemple_rel_dict)
lfs_instances

,lf,lf_name,source,target,form,merged,propform,keywords,values,kw_context
0,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:43890,NaN,0,[X] ~ Y,à,en direction,C’est comme de changer d’avion à Milan pour al...
1,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:41336,NaN,0,[X] ~ Y,à,sur,C’est comme de changer d’avion à Milan pour al...
2,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:29809,NaN,0,[X] ~ Y,à,vers,C’est comme de changer d’avion à Milan pour al...
3,ls:fr:lf:3,Syn_∩,ls:fr:node:26163,ls:fr:node:32392,NaN,0,[X] ~ de Y,à propos,au sujet,"Le docteur Ricci, que maman m’avait mené voir ..."
4,ls:fr:lf:3,Syn_∩,ls:fr:node:26163,ls:fr:node:32518,NaN,0,[X] ~ de Y,à propos,concernant,"Le docteur Ricci, que maman m’avait mené voir ..."
...,...,...,...,...,...,...,...,...,...,...
65484,ls:fr:lf:455,S_1CausFunc_0,ls:fr:node:56951,ls:fr:node:37921,NaN,0,~ utilisé par X pour Y,solaire,soleil,"Le plus intéressant, rapide à mettre en place ..."
65485,ls:fr:lf:239,S_0Real_1,ls:fr:node:56951,ls:fr:node:56952,NaN,1,~ utilisé par X pour Y,solaire,solaire,"Le plus intéressant, rapide à mettre en place ..."
65486,ls:fr:lf:1089,Gener_⊃,ls:fr:node:56952,ls:fr:node:27929,NaN,0,~ où travaillent les X,solaire,industrie,Le 【solaire】 se développe rapidement : le taux...
65487,ls:fr:lf:68,S_med,ls:fr:node:56952,ls:fr:node:56950,NaN,0,~ où travaillent les X,solaire,énergie solaire,Le 【solaire】 se développe rapidement : le taux...


In [28]:
node_to_propform = propforms['actant_form'].to_dict()
lfs_instances['propform'] = lfs_instances['source'].map(node_to_propform)
lfs_instances

,lf,lf_name,source,target,form,merged,propform,keywords,values,kw_context
0,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:43890,NaN,0,[X] ~ Y,à,en direction,C’est comme de changer d’avion à Milan pour al...
1,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:41336,NaN,0,[X] ~ Y,à,sur,C’est comme de changer d’avion à Milan pour al...
2,ls:fr:lf:3,Syn_∩,ls:fr:node:26162,ls:fr:node:29809,NaN,0,[X] ~ Y,à,vers,C’est comme de changer d’avion à Milan pour al...
3,ls:fr:lf:3,Syn_∩,ls:fr:node:26163,ls:fr:node:32392,NaN,0,[X] ~ de Y,à propos,au sujet,"Le docteur Ricci, que maman m’avait mené voir ..."
4,ls:fr:lf:3,Syn_∩,ls:fr:node:26163,ls:fr:node:32518,NaN,0,[X] ~ de Y,à propos,concernant,"Le docteur Ricci, que maman m’avait mené voir ..."
...,...,...,...,...,...,...,...,...,...,...
65484,ls:fr:lf:455,S_1CausFunc_0,ls:fr:node:56951,ls:fr:node:37921,NaN,0,~ utilisé par X pour Y,solaire,soleil,"Le plus intéressant, rapide à mettre en place ..."
65485,ls:fr:lf:239,S_0Real_1,ls:fr:node:56951,ls:fr:node:56952,NaN,1,~ utilisé par X pour Y,solaire,solaire,"Le plus intéressant, rapide à mettre en place ..."
65486,ls:fr:lf:1089,Gener_⊃,ls:fr:node:56952,ls:fr:node:27929,NaN,0,~ où travaillent les X,solaire,industrie,Le 【solaire】 se développe rapidement : le taux...
65487,ls:fr:lf:68,S_med,ls:fr:node:56952,ls:fr:node:56950,NaN,0,~ où travaillent les X,solaire,énergie solaire,Le 【solaire】 se développe rapidement : le taux...


## Calculer le score levenstein

In [ ]:
import Levenshtein
def get_levenshtein_distance(s1, s2):
    """
    Calculate the Levenshtein distance between two strings.
    """
    return Levenshtein.distance(s1, s2)
def get_levenshtein_ratio(s1, s2):
    """
    Calculate the Levenshtein ratio between two strings.
    """
    return Levenshtein.ratio(s1, s2)

In [30]:
lfs_instances['levenshtein_distance'] = lfs_instances.apply(
	lambda row: get_levenshtein_distance(row['keywords'], row['values']), axis=1
)
lfs_instances['levenshtein_ratio'] = lfs_instances.apply(
	lambda row: get_levenshtein_ratio(row['keywords'], row['values']), axis=1
)
lfs_instances['levenshtein_distance'] = lfs_instances['levenshtein_distance'].astype(int)
lfs_instances['levenshtein_ratio'] = lfs_instances['levenshtein_ratio'].astype(float)
print(lfs_instances['levenshtein_distance'].describe())
print(lfs_instances['levenshtein_ratio'].describe())

count    65489.000000
mean         6.332331
std          3.795735
min          0.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         64.000000
Name: levenshtein_distance, dtype: float64
count    65489.000000
mean         0.429185
std          0.262486
min          0.000000
25%          0.222222
50%          0.363636
75%          0.625000
max          1.000000
Name: levenshtein_ratio, dtype: float64


In [74]:
ex_mapping = exemple_rel.set_index('content')['node'].to_dict()
len(ex_mapping)

20776

In [75]:
print(lfs_instances.shape)
print(nodes_entry.shape)
print(exemples.shape)

(65489, 10)
(29783, 9)
(32026, 8)


In [77]:
lfs_instances = lfs_instances.drop(columns=['source', 'target'])

In [78]:
lfs_instances.dropna(subset=['propform'], inplace=True)

In [80]:
lfs_instances = lfs_instances.applymap(lambda x: str(x).replace('\u2028', ' ').replace('\u2029', ' ') if isinstance(x, str) else x)
lfs_instances.shape

/tmp/ipykernel_380892/2966231895.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lfs_instances = lfs_instances.applymap(lambda x: str(x).replace('\u2028', ' ').replace('\u2029', ' ') if isinstance(x, str) else x)


(55571, 8)

In [ ]:
# lfs_instances.to_csv('outputs/lfs_instances_propform_cleaned.csv', index=False)

In [13]:
# Exemple of instances
import pandas as pd
lfs_instances = pd.read_csv('../data/processed/lfs-slctd.csv')
lfs_instances.head()
lfs_instances_sp = lfs_instances[["index","lf_id", "lf_name", "keyword", "value", "kw_propform", "vl_propform", "kw_context"]]
ex = lfs_instances_sp[lfs_instances_sp['index'].isin([415, 42, 11516, 3806, 4961])].drop(columns=['index'])
ex.to_csv('../data/interim/lfs_instances_examples.csv', index=False)